# Add your own dataset to RadHarmony

This notebook shows, step by step, how to add a new radiology dataset. Its code cells are a template that you
can copy. Each line that you must change for your dataset has the comment `MUST CHANGE`.

---

## The 2 parts

Each dataset has 2 classes:

| Part | Class | Its job |
|---|---|---|
| **Harmonizer** | `MyDatasetHarmonizer(BaseHarmonizer)` | Reads the source files and returns the harmonized table, one row for each image |
| **Dataset class** | `MyDataset(BaseRadiologicalDataset)` | Gives the samples. It sets `LABEL_COLS`, `SUPPORTED_OUTPUTS` and `_HARMONIZER_CLS`, and its `_get_harmonized_df()` runs the harmonizer |

The base classes do the rest: the MONAI transforms, the disk cache, the split by patient, the folds and the
image checks. `@register_dataset("<key>")` adds the dataset class to the registry.

Do not change the base classes (`BaseHarmonizer`, `BaseRadiologicalDataset` and the transforms). All datasets
share them, so a change there can change the results of all datasets. Put the new logic in the classes of your
dataset. A base code change needs the approval of a maintainer first
([D-20](https://f10409.github.io/RadHarmony/design.html#d-20-base-code-needs-approval)).

---

## Steps

| Step | What you do |
|---|---|
| 0 | Set the paths (and make the example CSV files) |
| 1 | Write the harmonizer |
| 2 | Write the dataset class |
| 3 | Check the harmonized table |
| 4 | Split the data and look at a sample |
| 5 | Add augmentation |
| 6 | Use the registry |
| 7 | Save and reuse the harmonized table |
| 8 | Move the dataset into the package |

---

## What you need before you start

| Item | Example |
|---|---|
| The image folder | `/data/my_dataset/images/` |
| One or more CSV files with the labels and other data (for example age or view) | `images.csv`, `labels.csv` |
| How the CSV names each image | a file name column, or a path relative to the image folder |
| The patient ID | `PatientID`, `subject_id` |
| The study ID | `StudyInstanceUID`, `accession_number` |
| The label columns | `["Pneumonia", "Pleural Effusion"]` |
| The modality | 2-D (X-ray, DICOM, PNG) or 3-D (CT, NIfTI) |

---

## Step 0: Set the paths

This notebook uses the Montgomery County CXR images as the "new" dataset, so that you can run each cell. The
next cell writes 2 small CSV files for these images: `images.csv` (one row for each image) and `labels.csv`
(the TB label of each image). To download the images (about 600 MB, no account), see
[Get the data: Montgomery County CXR](https://f10409.github.io/RadHarmony/how-to/get-the-data.html#montgomery-county-cxr).

For your own dataset, set the paths to your image folder and your CSV files. Then delete the example part of
the cell.

In [ ]:
import os

import pandas as pd

# MUST CHANGE: the image folder and the CSV files of your dataset.
BASE_IMAGE_DIR = "/path/to/MontgomerySet/CXR_png/"
DATA_DIR = "./my_dataset"
CSV_PATH = os.path.join(DATA_DIR, "images.csv")
LABEL_CSV_PATH = os.path.join(DATA_DIR, "labels.csv")
CACHE_DIR = "./cache/my_dataset"

# Example only: write the 2 CSV files for the Montgomery images. Delete this part for your dataset.
# A file name such as MCUCXR_0001_0.png holds the image number (0001) and the TB label (0 or 1).
os.makedirs(DATA_DIR, exist_ok=True)
files = sorted(f for f in os.listdir(BASE_IMAGE_DIR) if f.endswith(".png"))
# The example counts each image as a patient (138 patients). The real Montgomery table has 135 patients.
# pd.read_csv reads "0001" as the number 1, so the example writes the numbers as int.
numbers = [int(f.split("_")[1]) for f in files]
images = pd.DataFrame({"PatientID": numbers, "StudyID": numbers, "FileName": files, "ViewPosition": "PA"})
labels = pd.DataFrame({"PatientID": numbers, "StudyID": numbers, "Tuberculosis": [int(f[-5]) for f in files]})
images.to_csv(CSV_PATH, index=False)
labels.to_csv(LABEL_CSV_PATH, index=False)
print(images.head(3))
print(labels["Tuberculosis"].value_counts().to_dict())

---

## Step 1: Write the harmonizer

Make a subclass of `BaseHarmonizer`. `harmonize()` reads the source files with `_read_source()` into
`self.df`, and then runs the build steps. The default `_read_source()` reads the CSV of `csv_path`. Write
these 3 steps:

| Step | What it does |
|---|---|
| `_build_patient_id()` | Sets `self.df["patient_id"]` |
| `_build_study_id()` | Sets `self.df["study_id"]` |
| `_build_image_path()` | Sets `self.df["image_path"]`, a path relative to `base_image_dir` |

`base_image_dir` is the deepest folder that all image files share. Example: for `/data/cxr14/images_001/a.png`
and `/data/cxr14/images_002/b.png`, `base_image_dir` is `/data/cxr14` and `image_path` is `images_001/a.png`
([D-04](https://f10409.github.io/RadHarmony/design.html#d-04-image-paths-are-relative-to-base_image_dir)).
When the dataset has official splits, each split gets its own class and its own folder, for example `train/`
and `test/`
([D-09](https://f10409.github.io/RadHarmony/design.html#d-09-official-splits-are-separate-classes)).

Class attributes set the optional columns. The base steps read them:

| Attribute | What it does |
|---|---|
| `LABEL_COLS` | The label columns, as the source names them. The base class renames them to snake case and sorts them |
| `LABEL_JOIN_COLS` | The columns that join a separate label CSV (`label_csv_path`) to the table |
| `VIEW_POSITION_SOURCE_COL` / `VIEW_POSITION_JOIN_COLS` | The view column (renamed to `view_position`), and the join columns of a separate view CSV |
| `MASK_SOURCE_COL` / `MASK_JOIN_COLS` | The mask column (renamed to `mask_path`), and the join columns of a separate mask CSV |
| `BBOX_SOURCE_COL` / `BBOX_JOIN_COLS` | The box column (renamed to `bbox`), and the join columns of a separate box CSV |
| `REPORT_PATH_COL` / `REPORT_JOIN_COLS` | The column of the report CSV with the path of each report file, and its join columns. The base class reads the text into `report` |
| `AGE_SOURCE_COL` / `AGE_SOURCE_UNIT` | The age column and its unit. The base class makes `age_years` |
| `SEX_SOURCE_COL` | The sex column. The base class makes `sex`: `"M"`, `"F"` or NaN |
| `REG_COLS` | The regression targets (continuous values), for `output_reg` |
| `EXTRA_OUTPUT_COLS` | Other source columns to keep in the table |

Give the path of each separate CSV with its own argument of `BaseHarmonizer.__init__`: `label_csv_path`,
`view_position_csv_path`, `mask_csv_path`, `bbox_csv_path` and `report_csv_path`. To clean a CSV before the
join, override its step: `_preprocess_label_df`, `_preprocess_view_position_df`, `_preprocess_mask_df`,
`_preprocess_bbox_df` or `_preprocess_report_df`. To decode masks that the CSV keeps as text (for example
RLE), override `_decode_mask`.

Rules of the harmonized table:

- One row for each image. A join CSV with more than one row for a key raises `ValueError`
  ([D-03](https://f10409.github.io/RadHarmony/design.html#d-03-one-row-per-image)).
- A label is 1 (positive), 0 (negative), -1 (uncertain) or NaN (not known)
  ([D-05](https://f10409.github.io/RadHarmony/design.html#d-05-labels)).
- Change the steps, not `harmonize()`. Override `harmonize()` only to add an option, for example
  `drop_uncertain`, or to add steps after the base steps. Then call `super().harmonize()` in it
  ([D-30](https://f10409.github.io/RadHarmony/design.html#d-30-a-harmonizer-changes-steps-not-harmonize)).
- When the source is not one CSV file, override `_read_source()`.
- `save()` keeps the arguments of the base class, and also `base_image_dir` and `dicom_base_dir`. If your
  `__init__` takes another argument, add it in `_harmonizer_init_snapshot()`.

In [ ]:
from radharmony.harmonizer.base import BaseHarmonizer


class MyDatasetHarmonizer(BaseHarmonizer):  # MUST CHANGE: the class name
    """Harmonize MyDataset: one row for each image."""

    # MUST CHANGE: the label columns, as the source files name them. "Tuberculosis" becomes "tuberculosis".
    LABEL_COLS = ["Tuberculosis"]
    # The columns of the harmonized table that join labels.csv to images.csv.
    LABEL_JOIN_COLS = ["patient_id", "study_id"]
    # The source column of the view. Use None if the source has no view.
    VIEW_POSITION_SOURCE_COL = "ViewPosition"

    def __init__(self, csv_path: str, label_csv_path: str = None, base_image_dir: str = None):
        super().__init__(csv_path=csv_path, label_csv_path=label_csv_path)
        self.base_image_dir = base_image_dir

    # The 3 required steps. self.df holds the rows of csv_path.
    def _build_patient_id(self) -> None:
        # astype(str): pd.read_csv reads an ID column of digits as numbers.
        self.df["patient_id"] = self.df["PatientID"].astype(str)  # MUST CHANGE

    def _build_study_id(self) -> None:
        self.df["study_id"] = self.df["StudyID"].astype(str)  # MUST CHANGE

    def _build_image_path(self) -> None:
        # MUST CHANGE: the path relative to base_image_dir (D-04).
        self.df["image_path"] = self.df["FileName"]

    # An optional step: clean the label CSV before the join.
    def _preprocess_label_df(self, label_df: pd.DataFrame) -> pd.DataFrame:
        """Rename the ID columns of the label CSV to the names of the harmonized table."""
        label_df = label_df.rename(columns={"PatientID": "patient_id", "StudyID": "study_id"})
        label_df["patient_id"] = label_df["patient_id"].astype(str)
        label_df["study_id"] = label_df["study_id"].astype(str)
        return label_df

---

## Step 2: Write the dataset class

Make a subclass of `BaseRadiologicalDataset`, and set:

1. **`LABEL_COLS`**: the snake case label names of the harmonized table.
2. **`SUPPORTED_OUTPUTS`**: the outputs that the dataset has. A flag that is not in this set gives a warning
   and no key ([D-11](https://f10409.github.io/RadHarmony/design.html#d-11-output-flags)).
3. **`_HARMONIZER_CLS`**: the harmonizer class. `harmonizer_path=` needs it to load a file from
   `MyDatasetHarmonizer.save()`.
4. **`_get_harmonized_df()`**: first, call `_try_resolve_preset_harmonized()`. If it returns a table, return
   this table. If it returns `None`, run the harmonizer.

Give `harmonized_df`, `harmonizer` and `harmonizer_path` to `super().__init__(...)`. Then a user can skip the
harmonize step with a table, a harmonizer object or a saved file. The base class does the rest: the
transforms, the cache and the splits.

In [ ]:
import torch

from radharmony.dataset import BaseRadiologicalDataset, RadiologyTransform2D
from radharmony.registry import register_dataset


@register_dataset("my_dataset")  # MUST CHANGE: a registry key that no other dataset has
class MyDataset(BaseRadiologicalDataset):  # MUST CHANGE: the class name
    """PyTorch dataset for MyDataset.

    Args:
        base_image_dir: The deepest folder that all image files share (D-04).
        csv_path: The CSV with one row for each image.
        label_csv_path: The CSV with the labels.
        transform: A MONAI transform. None gives the default 2-D steps (224 pixels).
        cache_dir: The cache folder. None turns off the disk cache.
        output_cls, output_mask, output_report, output_bbox, output_reg: The output flags (D-11).
        harmonized_df: A harmonized table to use, with no harmonize step.
        harmonizer: A harmonizer object with a harmonized table.
        harmonizer_path: A file from MyDatasetHarmonizer.save().
        dtype: The type of img.
    """

    LABEL_COLS = ["tuberculosis"]  # MUST CHANGE: the snake case label names
    # MUST CHANGE: add "mask", "bbox", "report" or "reg" if your dataset has them.
    SUPPORTED_OUTPUTS = frozenset({"cls"})
    _HARMONIZER_CLS = MyDatasetHarmonizer

    def __init__(
        self,
        base_image_dir: str,
        csv_path: str = None,
        label_csv_path: str = None,
        transform=None,
        cache_dir: str = "./cache",
        output_cls: bool = False,
        output_mask: bool = False,
        output_report: bool = False,
        output_bbox: bool = False,
        output_reg: bool = False,
        harmonized_df=None,
        harmonizer=None,
        harmonizer_path: str = None,
        dtype=torch.bfloat16,
    ):
        self._csv_path = csv_path
        self._label_csv_path = label_csv_path
        if transform is None:
            # Keep the key of each flag that the dataset supports. Use RadiologyTransform3D for CT.
            flags = {"cls": output_cls, "mask": output_mask, "report": output_report,
                     "bbox": output_bbox, "reg": output_reg}
            output_keys = {"img"} | {k for k, on in flags.items() if on and k in self.SUPPORTED_OUTPUTS}
            if "bbox" in output_keys:
                output_keys.add("bbox_labels")
            transform = RadiologyTransform2D(img_size=224, output_keys=output_keys, dtype=dtype).get_transform()
        super().__init__(
            base_image_dir=base_image_dir,
            transform=transform,
            cache_dir=cache_dir,
            output_cls=output_cls,
            output_mask=output_mask,
            output_report=output_report,
            output_bbox=output_bbox,
            output_reg=output_reg,
            harmonized_df=harmonized_df,
            harmonizer=harmonizer,
            harmonizer_path=harmonizer_path,
        )

    def _get_harmonized_df(self) -> pd.DataFrame:
        """Return the harmonized table: the preset table, or a new harmonize() result."""
        preset = self._try_resolve_preset_harmonized()
        if preset is not None:
            return preset
        # MUST CHANGE: give the other paths that your harmonizer needs.
        return MyDatasetHarmonizer(
            csv_path=self._csv_path,
            label_csv_path=self._label_csv_path,
            base_image_dir=self.base_image_dir,
        ).harmonize()

---

## Step 3: Check the harmonized table

Check the table of the harmonizer before you make a dataset. `verify_images()` checks that each image file is
on the disk. With `drop_missing=False`, it returns the rows with no file and does not drop them. See
[Dataset API: Harmonized table columns](https://f10409.github.io/RadHarmony/reference/api-datasets.html#harmonized-table-columns).

In [ ]:
h = MyDatasetHarmonizer(csv_path=CSV_PATH, label_csv_path=LABEL_CSV_PATH, base_image_dir=BASE_IMAGE_DIR)
df = h.harmonize()

print(f"Shape          : {df.shape}")
print(f"Unique patients: {df['patient_id'].nunique():,}")
print(f"Unique studies : {df['study_id'].nunique():,}")
print(f"Columns        : {df.columns.tolist()}")

# The required columns.
missing = [c for c in ["patient_id", "study_id", "image_path"] if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

# image_path is relative to base_image_dir (D-04).
assert not df["image_path"].map(os.path.isabs).any(), "image_path must be relative to base_image_dir"

# A label is 1, 0, -1 or NaN (D-05).
for col in MyDataset.LABEL_COLS:
    values = set(df[col].dropna().unique())
    assert values <= {0, 1, -1}, f"Label '{col}' has other values: {values}"

# Each image file is on the disk.
missing_files = h.verify_images(drop_missing=False)
assert missing_files.empty

print("Label prevalence:", df[MyDataset.LABEL_COLS].mean().round(3).to_dict())
df.head(3)

---

## Step 4: Split the data and look at a sample

### 4.1 Train and val split

`get_datasets(n_splits=N)` returns `(train_dataset, val_dataset)`. The val split gets about 1/N of the
patients. All images of one patient go to the same split. See
[Split data](https://f10409.github.io/RadHarmony/how-to/split-data.html).

In [ ]:
ds = MyDataset(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    cache_dir=CACHE_DIR,
    output_cls=True,
)

train_ds, val_ds = ds.get_datasets(n_splits=5)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

### 4.2 Look at a sample

Each sample is a dict. The `output_*` flags set its keys. See
[Dataset API: Data dict keys](https://f10409.github.io/RadHarmony/reference/api-datasets.html#data-dict-keys).

In [ ]:
sample = train_ds[0]

for key, val in sample.items():
    if isinstance(val, torch.Tensor):
        print(
            f"  {key:12s}  shape={tuple(val.shape)}  dtype={val.dtype}  "
            f"min={val.float().min():.3f}  max={val.float().max():.3f}"
        )
    else:
        print(f"  {key:12s}  {type(val).__name__}: {val}")

### 4.3 Show a sample

The next cell shows the image of the sample, with its positive labels in the title.

In [ ]:
import matplotlib.pyplot as plt

img = sample["img"].float().squeeze().numpy()  # (H, W) for a 2-D image

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
ax.axis("off")
positives = [MyDataset.LABEL_COLS[i] for i, v in enumerate(sample["cls"]) if v > 0]
ax.set_title(", ".join(positives) if positives else "no finding", fontsize=8)
plt.tight_layout()
plt.show()

---

## Step 5: Add augmentation

`RadiologyTransform2D` and `RadiologyTransform3D` have `with_*` methods that add augmentation steps after the
default steps. Set `output_keys` to all the keys of the flags that you use. If the transform drops the key of
a flag, the dataset class warns ([D-11](https://f10409.github.io/RadHarmony/design.html#d-11-output-flags)).

Give the augmentation to the train split only, with `train_transform=`. If you give no `val_transform=` and
the dataset transform has a random step, `get_datasets()` warns
([D-49](https://f10409.github.io/RadHarmony/design.html#d-49-a-val-split-with-random-steps-warns)).

[Transforms](https://f10409.github.io/RadHarmony/reference/transforms.html) lists all methods and the 3-D
steps. It also shows how to write a transform of your own. The
[`dataset_api_tour.ipynb`](dataset_api_tour.ipynb) notebook shows more examples (section 7).

In [ ]:
train_transform = (
    RadiologyTransform2D(img_size=224, output_keys={"img", "cls"})
    .with_flip(spatial_axis=1, prob=0.5)  # left-right flip
    .with_affine(translate_range=(10, 10), rotate_range=(0.17,), scale_range=(0.1, 0.1), prob=0.5)
    .with_intensity_jitter(shift=0.1, scale=0.1, prob=0.5)
    .get_transform()
)

# The val split keeps the default transform of the dataset, with no random step.
train_ds, val_ds = ds.get_datasets(n_splits=5, train_transform=train_transform)
print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
print(train_ds[0]["img"].shape)

---

## Step 6: Use the registry

`@register_dataset("my_dataset")` adds the class to the registry when the cell runs. Then `list_datasets()`
shows the key, and `resolve_dataset("my_dataset")` returns the class, as for the datasets of the
[Dataset catalog](https://f10409.github.io/RadHarmony/reference/datasets/index.html).

A key can be in the registry only one time. If you run the cell of step 2 a second time, `register_dataset`
raises `ValueError`. To fix this, restart the kernel. Then run all cells again.

In [ ]:
from radharmony.registry import list_datasets, resolve_dataset

print("my_dataset" in list_datasets())

DatasetCls = resolve_dataset("my_dataset")  # MUST CHANGE: your registry key
train_ds, val_ds = DatasetCls(
    base_image_dir=BASE_IMAGE_DIR,
    csv_path=CSV_PATH,
    label_csv_path=LABEL_CSV_PATH,
    cache_dir=CACHE_DIR,
    output_cls=True,
).get_datasets(n_splits=5)

print(DatasetCls.__name__, f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")

---

## Step 7: Save and reuse the harmonized table

A large dataset can take a long time to harmonize. Save the harmonizer one time with `save()`. Then give the
file to the dataset class with `harmonizer_path=`. This needs `_HARMONIZER_CLS`. See
[Save and reuse a harmonized table](https://f10409.github.io/RadHarmony/how-to/save-and-reuse.html).

In [ ]:
SAVED = os.path.join(DATA_DIR, "my_dataset_harmonizer.pkl")
h.save(SAVED)

ds_saved = MyDataset(base_image_dir=BASE_IMAGE_DIR, harmonizer_path=SAVED, cache_dir=None, output_cls=True)
print(len(ds_saved.get_harmonized_df()), ds_saved.LABEL_COLS)

h2 = MyDatasetHarmonizer.load_from_saved(SAVED)
print(h2.base_image_dir == BASE_IMAGE_DIR)

---

## Step 8: Move the dataset into the package

A notebook class is good for a test. To add the dataset to RadHarmony for all users, put the 2 classes in the
package. Also add their config, their integrity check and their wiki page.
[Add a dataset](https://f10409.github.io/RadHarmony/how-to/add-a-dataset.html) lists all the files.

- The harmonizer goes to `radharmony/harmonizer/<name>/<name>.py`, and the dataset class goes to
  `radharmony/dataset/<name>/<name>.py`.
- Do not write the real paths of your data in the package files. The user gives them as arguments.
- To show the dataset in the app, add a build function and a `DatasetConfig` entry to `app.py`. Or write a
  plugin module. Then start the app with `RADHARMONY_APP_PLUGINS=<module> python app.py`. See
  [Add a dataset: Show the dataset in the app](https://f10409.github.io/RadHarmony/how-to/add-a-dataset.html#2-show-the-dataset-in-the-app).

In Claude Code, the
[add-dataset skill](https://github.com/f10409/RadHarmony/blob/main/.claude/skills/add-dataset/SKILL.md) does
all the steps with you.